In [ ]:
# -*- coding: utf-8 -*-
"""
Constraint-Driven Warm-Freeze (CDWF) - Spike Detection

Applies CDWF to spike attack detection in photovoltaic MPPT systems
using a 1D ResNet architecture.
"""

import os
import gc
import math
import random
import time
import copy

from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


# ================================================================
# GPU POWER MONITORING
# ================================================================
try:
    import pynvml

    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print("GPU power monitoring enabled")
except Exception:
    NVML_AVAILABLE = False
    _nvml_handle = None
    print("GPU power monitoring unavailable")


def read_gpu_power_w():
    if not NVML_AVAILABLE:
        return 0.0
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_nvml_handle) / 1000.0
    except Exception:
        return 0.0


# ================================================================
# CONFIGURATION
# ================================================================
EXPERIMENT_NAME = "spike_resnet50_cdwf"
SEED = 42

# Data paths
NORMAL_DIR = "/normal_vs_spike/data/normal"
ATTACK_DIR = "/normal_vs_spike/data/attack"
BIAS_CKPT = "best_resnet1d_basicblock_bias.pth"

# Model
MODEL_DEPTH = 50

# Training
BATCH_SIZE = 64
NUM_WORKERS = 2
MAX_LR = 3e-4
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0

# Constraint-driven configuration
EPOCHS_FULLFT = 10

# Constraint settings
CONSTRAINT_TYPE = "params"
TARGET_SPEEDUP = 2.0
TARGET_PARAMS_PCT = 0.01
TARGET_ENERGY_REDUCTION = 2.5
MAX_TRAINABLE_FRACTION = 0.01

# Warm-start and fine-tuning
EPOCHS_WARMSTART_BRIEF = 3
EPOCHS_FINETUNE = 7

# LoRA options
LORA_RANKS = [4, 6, 8] # Could hold one or more LoRA ranks
LORA_ALPHA = 1.0

# Data
NUM_CLASSES = 2
FULLFT_CSV = "spike_resnet50_fullft_results.csv"
RESULT_CSV = "spike_resnet50_cdwf_results.csv"

print(f"\n{'=' * 70}")
print("  CDWF - SPIKE DETECTION")
print(f"{'=' * 70}")
print(f"Model: ResNet-{MODEL_DEPTH} 1D")
print("Task: Normal vs Spike Attack Detection")
print(f"Constraint: {CONSTRAINT_TYPE.upper()}")
if CONSTRAINT_TYPE == "speedup":
    print(f"Target: {TARGET_SPEEDUP:.1f}x speedup")
elif CONSTRAINT_TYPE == "params":
    print(f"Target: {TARGET_PARAMS_PCT * 100:.1f}% trainable params")
print(f"Batch size: {BATCH_SIZE}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"{'=' * 70}\n")


# ================================================================
# REPRODUCIBILITY
# ================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")


# ================================================================
# DATASET
# ================================================================
class SpikeDataset(Dataset):
    """Dataset for spike attack detection."""

    def __init__(self, X, M, y):
        self.X = X.astype(np.float32)
        self.M = M.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        vals = self.X[idx]
        msk = self.M[idx]

        valid = msk > 0.5
        if valid.any():
            v = vals[valid]
            mu = float(v.mean())
            sd = float(v.std() + 1e-6)
            vals = (vals - mu) / sd
        else:
            vals = vals * 0.0

        vals = torch.from_numpy(vals[None, :])
        msk = torch.from_numpy(msk[None, :])
        lab = torch.tensor(self.y[idx], dtype=torch.long)
        return vals, msk, lab


def load_spike_data_jolt_format(normal_dir, attack_dir):
    """Load spike data with padding and masking."""
    import re

    re_normal = re.compile(r"^n_(\d+)\.npy$", re.IGNORECASE)
    re_attack = re.compile(r"^a_(\d+)\.npy$", re.IGNORECASE)

    def map_indices(dir_path, pattern):
        mapping = {}
        for fn in os.listdir(dir_path):
            match = pattern.match(fn)
            if match:
                mapping[int(match.group(1))] = fn
        return mapping

    def safe_load_array(path):
        try:
            arr = np.load(path, allow_pickle=False)
            return np.asarray(arr).ravel().astype(np.float32)
        except Exception as exc:
            print(f"[load warning] {path} -> {exc}")
            return None

    n_map = map_indices(normal_dir, re_normal)
    a_map = map_indices(attack_dir, re_attack)
    common_idx = sorted(set(n_map.keys()) & set(a_map.keys()))

    print(f"  Matched indices: {len(common_idx)}")

    samples, labels, pair_ids = [], [], []

    for idx in tqdm(common_idx, desc="Loading"):
        n_path = os.path.join(normal_dir, n_map[idx])
        a_path = os.path.join(attack_dir, a_map[idx])

        n_arr = safe_load_array(n_path)
        a_arr = safe_load_array(a_path)
        if n_arr is None or a_arr is None:
            continue

        samples.append(n_arr)
        labels.append(0)
        pair_ids.append(idx)

        samples.append(a_arr)
        labels.append(1)
        pair_ids.append(idx)

    print(f"  Total samples: {len(labels)}")

    lengths = [len(s) for s in samples]
    max_len = max(lengths)

    X_full = np.full((len(samples), max_len), np.nan, np.float32)
    for i, s in enumerate(samples):
        X_full[i, :len(s)] = s

    y_full = np.asarray(labels, dtype=np.int64)
    pair_ids = np.asarray(pair_ids, dtype=np.int64)

    mask_full = ~np.isnan(X_full)
    X_filled = np.where(mask_full, X_full, 0.0).astype(np.float32)
    mask_full = mask_full.astype(np.float32)

    return X_filled, mask_full, y_full, pair_ids


print("Loading spike detection data...")
X_filled, mask_full, y_full, pair_ids = load_spike_data_jolt_format(NORMAL_DIR, ATTACK_DIR)

TRAIN_FRACTION = 0.80
VAL_FRACTION_WITHIN_TRAIN = 0.20

unique_pairs = sorted(set(pair_ids.tolist()))
num_pairs = len(unique_pairs)
print(f"  Total pairs: {num_pairs}")

rng = np.random.RandomState(SEED)
perm = rng.permutation(num_pairs)
ordered_pairs = [unique_pairs[i] for i in perm]

num_train_pairs_total = int(math.floor(num_pairs * TRAIN_FRACTION))
num_val_pairs = int(math.floor(num_train_pairs_total * VAL_FRACTION_WITHIN_TRAIN))

train_pairs_all = ordered_pairs[:num_train_pairs_total]
test_pairs = ordered_pairs[num_train_pairs_total:]

if num_val_pairs > 0:
    val_pairs = train_pairs_all[:num_val_pairs]
    train_pairs = train_pairs_all[num_val_pairs:]
else:
    val_pairs = []
    train_pairs = train_pairs_all

train_mask = np.isin(pair_ids, train_pairs)
val_mask = np.isin(pair_ids, val_pairs)
test_mask = np.isin(pair_ids, test_pairs)

X_train, M_train, y_train = X_filled[train_mask], mask_full[train_mask], y_full[train_mask]
X_val, M_val, y_val = X_filled[val_mask], mask_full[val_mask], y_full[val_mask]
X_test, M_test, y_test = X_filled[test_mask], mask_full[test_mask], y_full[test_mask]

print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}\n")

train_dataset = SpikeDataset(X_train, M_train, y_train)
val_dataset = SpikeDataset(X_val, M_val, y_val)
test_dataset = SpikeDataset(X_test, M_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ================================================================
# RESNET1D WITH MASKED GLOBAL AVERAGE POOLING
# ================================================================
def pick_gn_groups(channels):
    for groups in [32, 16, 8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        groups = pick_gn_groups(planes)

        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(groups, planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(groups, planes)

        self.down = None
        if stride != 1 or in_planes != planes:
            self.down = nn.Sequential(
                nn.Conv1d(in_planes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(pick_gn_groups(planes), planes),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(x)
        out = self.relu(out + identity)
        return out


class ResNet1D(nn.Module):
    def __init__(self, base=64, use_maxpool=False, num_classes=2):
        super().__init__()
        self.use_maxpool = use_maxpool

        self.conv1 = nn.Conv1d(1, base, 7, stride=2, padding=3, bias=False)
        self.gn1 = nn.GroupNorm(pick_gn_groups(base), base)
        self.relu = nn.ReLU(inplace=True)

        if use_maxpool:
            self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)

        self.inplanes = base
        self.layer1 = self._make_layer(base, blocks=2, stride=1)
        self.layer2 = self._make_layer(base * 2, blocks=2, stride=2)
        self.layer3 = self._make_layer(base * 4, blocks=2, stride=2)
        self.layer4 = self._make_layer(base * 8, blocks=2, stride=2)

        self.fc = nn.Linear(base * 8, num_classes)

        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.GroupNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def _make_layer(self, planes, blocks, stride):
        layers = [BasicBlock1D(self.inplanes, planes, stride)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.inplanes, planes, 1))
        return nn.Sequential(*layers)

    def forward(self, x, mask=None):
        x = self.relu(self.gn1(self.conv1(x)))
        if self.use_maxpool:
            x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        if mask is not None:
            pooled_mask = F.adaptive_max_pool1d(mask, x.size(-1))
            weighted_sum = (x * pooled_mask).sum(dim=-1)
            denom = pooled_mask.sum(dim=-1).clamp_min(1.0)
            x = weighted_sum / denom
        else:
            x = F.adaptive_avg_pool1d(x, 1).squeeze(-1)

        x = self.fc(x)
        return x


def resnet50_1d(base=64, use_maxpool=False, num_classes=2):
    return ResNet1D(base=base, use_maxpool=use_maxpool, num_classes=num_classes)


# ================================================================
# LORA FOR 1D CONVOLUTIONS
# ================================================================
class LoRAConv1d(nn.Module):
    """LoRA wrapper for Conv1d."""

    def __init__(self, base_conv, rank=1, alpha=1.0):
        super().__init__()
        self.base = base_conv
        for param in self.base.parameters():
            param.requires_grad = False

        self.rank = rank
        self.alpha = alpha

        in_ch = base_conv.in_channels
        out_ch = base_conv.out_channels
        kernel_size = base_conv.kernel_size[0]
        stride = base_conv.stride[0]
        padding = base_conv.padding[0]

        self.lora_A = nn.Conv1d(
            in_ch,
            rank,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.lora_B = nn.Conv1d(
            rank,
            out_ch,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )

        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.lora_B(self.lora_A(x))
        return base_out + (self.alpha / self.rank) * lora_out


# ================================================================
# MODEL UTILITIES
# ================================================================
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable


def get_block_dict(model):
    blocks = {}
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for i, block in enumerate(layer):
            blocks[f"{layer_name}_{i}"] = block
    return blocks


def get_block_param_count(block):
    return sum(p.numel() for p in block.parameters())


# ================================================================
# TRAINING FUNCTIONS
# ================================================================
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, scheduler, device, power_log=None):
    """Run one training epoch with optional power logging."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for vals, masks, targets in tqdm(loader, desc="Train", leave=False):
        if power_log is not None:
            now = time.time()
            power_log["energy_J"] += read_gpu_power_w() * (now - power_log["last_t"])
            power_log["last_t"] = now

        vals, masks, targets = vals.to(device), masks.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(vals, masks)
        loss = criterion(outputs, targets)
        loss.backward()

        if CLIP_NORM:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    return total_loss / len(loader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for vals, masks, targets in tqdm(loader, desc="Eval", leave=False):
        vals, masks, targets = vals.to(device), masks.to(device), targets.to(device)
        outputs = model(vals, masks)
        loss = criterion(outputs, targets)

        probs = torch.softmax(outputs, dim=1)

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(targets.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    acc = 100.0 * correct / total
    auc = roc_auc_score(all_labels, all_probs) * 100.0

    return total_loss / len(loader), acc, auc


# ================================================================
# PREDICTION AND OPTIMIZATION
# ================================================================
def compute_block_features(model, val_loader, device, num_batches=10):
    print("\n[Feature Extraction] Analyzing block characteristics...")

    model.eval()
    blocks = get_block_dict(model)

    features = {
        name: {
            "grad_mean": 0.0,
            "grad_std": 0.0,
            "grad_samples": [],
            "param_count": get_block_param_count(block),
            "layer_depth": int(name.split("_")[0][-1]),
        }
        for name, block in blocks.items()
    }

    for batch_idx, (vals, masks, targets) in enumerate(val_loader):
        if batch_idx >= num_batches:
            break

        vals, masks, targets = vals.to(device), masks.to(device), targets.to(device)
        model.zero_grad()

        outputs = model(vals, masks)
        loss = criterion(outputs, targets)
        loss.backward()

        for name, block in blocks.items():
            grad_norm = 0.0
            for param in block.parameters():
                if param.grad is not None:
                    grad_norm += param.grad.norm().item() ** 2
            grad_norm = math.sqrt(grad_norm)
            features[name]["grad_samples"].append(grad_norm)

    for name in features:
        samples = features[name]["grad_samples"]
        features[name]["grad_mean"] = np.mean(samples)
        features[name]["grad_std"] = np.std(samples)
        del features[name]["grad_samples"]

    return features


def build_predictor(block_features, warmstart_acc, fullft_acc):
    print("\n[Predictive Model] Building accuracy predictor...")

    total_grad = sum(f["grad_mean"] for f in block_features.values())

    block_importance = {}
    for name, feats in block_features.items():
        block_importance[name] = feats["grad_mean"] / total_grad

    max_gain = fullft_acc - warmstart_acc

    print(f"  Warm-start acc: {warmstart_acc:.2f}%")
    print(f"  Full_ft acc: {fullft_acc:.2f}%")
    print(f"  Maximum gain: {max_gain:.2f}%")
    print(f"  Total blocks: {len(block_features)}")

    return block_importance, max_gain, warmstart_acc


def predict_accuracy(block_importance, max_gain, warmstart_acc, kept_blocks, lora_blocks, lora_rank):
    kept_gain = sum(block_importance[b] for b in kept_blocks)
    lora_efficiency = min(0.5, lora_rank / 8.0)
    lora_gain = sum(block_importance[b] * lora_efficiency for b in lora_blocks)

    total_gain_fraction = kept_gain + lora_gain

    if max_gain > 10.0:
        epoch_discount = 0.4
    elif max_gain > 8.0:
        epoch_discount = 0.7
    else:
        epoch_discount = 0.9

    predicted_acc = warmstart_acc + (total_gain_fraction * max_gain * epoch_discount)
    return predicted_acc


def estimate_config_cost(model, kept_blocks, lora_blocks, lora_rank, warmstart_epochs, finetune_epochs, fullft_time):
    """
    Estimate training time using fixed overhead, trainable fraction,
    and LoRA overhead.
    """
    blocks = get_block_dict(model)
    total_params, _, _ = count_params(model)

    trainable = 0
    trainable += sum(p.numel() for p in model.fc.parameters())

    for name in kept_blocks:
        trainable += get_block_param_count(blocks[name])

    for name in lora_blocks:
        block = blocks[name]
        if hasattr(block, "conv2"):
            conv = block.conv2
            in_ch = conv.in_channels
            out_ch = conv.out_channels
            kernel_size = conv.kernel_size[0]
            trainable += lora_rank * (in_ch * kernel_size + out_ch)

    trainable_fraction = trainable / total_params

    fixed_fraction = 0.55
    variable_fraction = 0.45

    warmstart_time = (warmstart_epochs / EPOCHS_FULLFT) * fullft_time
    finetune_base = (finetune_epochs / EPOCHS_FULLFT) * fullft_time
    finetune_time = finetune_base * (
        fixed_fraction + variable_fraction * trainable_fraction
    )

    n_lora_blocks = len(lora_blocks)

    if lora_rank == 1:
        lora_overhead_per_block = 0.15
    elif lora_rank == 2:
        lora_overhead_per_block = 0.10
    elif lora_rank == 4:
        lora_overhead_per_block = 0.05
    else:
        lora_overhead_per_block = 0.02

    if n_lora_blocks > 0:
        total_lora_overhead = 1.0 + (lora_overhead_per_block * n_lora_blocks)
        finetune_time *= total_lora_overhead

    apply_overhead = 3.0

    total_time = warmstart_time + finetune_time + apply_overhead
    speedup = fullft_time / total_time

    return {
        "trainable_params": trainable,
        "trainable_fraction": trainable_fraction,
        "estimated_time": total_time,
        "speedup": speedup,
    }


def solve_optimal_architecture(
    block_features,
    block_importance,
    max_gain,
    warmstart_acc,
    model,
    fullft_acc,
    fullft_time,
    constraint_type,
    target_value,
):
    print("\n[Optimization] Solving for optimal architecture...")
    print(f"  Constraint: {constraint_type} = {target_value}")
    print(f"  Search space: {len(block_features)} blocks x {len(LORA_RANKS)} LoRA ranks")

    block_names = list(block_features.keys())
    n_blocks = len(block_names)

    sorted_blocks = sorted(
        block_names,
        key=lambda b: block_importance[b],
        reverse=True,
    )

    best_config = None
    best_score = -float("inf")

    for n_keep in range(0, n_blocks + 1):
        kept = sorted_blocks[:n_keep]
        frozen = sorted_blocks[n_keep:]

        for lora_rank in LORA_RANKS:
            pred_acc = predict_accuracy(
                block_importance,
                max_gain,
                warmstart_acc,
                kept,
                frozen,
                lora_rank,
            )

            cost = estimate_config_cost(
                model,
                kept,
                frozen,
                lora_rank,
                EPOCHS_WARMSTART_BRIEF,
                EPOCHS_FINETUNE,
                fullft_time,
            )

            constraint_met = False
            if constraint_type == "speedup":
                constraint_met = (
                    cost["speedup"] >= target_value
                    and cost["trainable_fraction"] <= MAX_TRAINABLE_FRACTION
                )
            elif constraint_type == "params":
                constraint_met = cost["trainable_fraction"] <= target_value
            elif constraint_type == "energy":
                constraint_met = (
                    cost["speedup"] >= target_value
                    and cost["trainable_fraction"] <= MAX_TRAINABLE_FRACTION
                )
            else:
                constraint_met = (
                    cost["speedup"] >= 2.0
                    and cost["trainable_fraction"] <= 0.05
                )

            if not constraint_met:
                continue

            score = pred_acc

            if score > best_score:
                best_score = score
                best_config = {
                    "kept_blocks": kept.copy(),
                    "lora_blocks": frozen.copy(),
                    "lora_rank": lora_rank,
                    "predicted_acc": pred_acc,
                    "trainable_params": cost["trainable_params"],
                    "trainable_fraction": cost["trainable_fraction"],
                    "estimated_speedup": cost["speedup"],
                    "estimated_time": cost["estimated_time"],
                }

    if best_config is None:
        print("  No configuration meets the constraint. Using fallback...")
        kept = sorted_blocks[:2]
        frozen = sorted_blocks[2:]
        lora_rank = 1

        pred_acc = predict_accuracy(
            block_importance,
            max_gain,
            warmstart_acc,
            kept,
            frozen,
            lora_rank,
        )
        cost = estimate_config_cost(
            model,
            kept,
            frozen,
            lora_rank,
            EPOCHS_WARMSTART_BRIEF,
            EPOCHS_FINETUNE,
            fullft_time,
        )

        best_config = {
            "kept_blocks": kept,
            "lora_blocks": frozen,
            "lora_rank": lora_rank,
            "predicted_acc": pred_acc,
            "trainable_params": cost["trainable_params"],
            "trainable_fraction": cost["trainable_fraction"],
            "estimated_speedup": cost["speedup"],
            "estimated_time": cost["estimated_time"],
        }

    print("\n[Optimal Configuration Found]")
    print(f"  Keep trainable: {len(best_config['kept_blocks'])} blocks")
    print(
        f"  Freeze + LoRA rank-{best_config['lora_rank']}: "
        f"{len(best_config['lora_blocks'])} blocks"
    )
    print(f"  Predicted accuracy: {best_config['predicted_acc']:.2f}%")
    print(f"  Estimated speedup: {best_config['estimated_speedup']:.2f}x")
    print(f"  Trainable params: {best_config['trainable_fraction'] * 100:.2f}%")

    return best_config


# ================================================================
# LOAD FULL_FT BASELINE FROM CSV
# ================================================================
print("=" * 70)
print("  LOADING SPIKE FULL_FT BASELINE")
print("=" * 70 + "\n")

if os.path.exists(FULLFT_CSV):
    print(f"Loading baseline from: {FULLFT_CSV}")
    fullft_df = pd.read_csv(FULLFT_CSV)

    fullft_auc = fullft_df["test_auc"].iloc[0]
    fullft_time = fullft_df["training_time_s"].iloc[0]
    fullft_energy = fullft_df["energy_J"].iloc[0]

    print("Loaded Full_FT baseline")
    print(f"  AUC: {fullft_auc:.2f}%")
    print(f"  Time: {fullft_time:.1f}s")
    print(f"  Energy: {fullft_energy / 1000:.2f}kJ\n")
else:
    print("Error: Full_FT baseline not found.")
    print(f"Expected file: {FULLFT_CSV}")
    print("Run the full fine-tuning baseline script first.\n")
    raise SystemExit(1)

if os.path.exists(BIAS_CKPT):
    print(f"Pre-trained checkpoint available: {BIAS_CKPT}\n")
else:
    print(f"Pre-trained checkpoint not found: {BIAS_CKPT}")
    print("Training will use random initialization.\n")


# ================================================================
# PHASE 2: CONSTRAINT-DRIVEN WARM-FREEZE
# ================================================================
print("=" * 70)
print("  PHASE 2: CONSTRAINT-DRIVEN WARM-FREEZE")
print("=" * 70 + "\n")

torch.cuda.empty_cache()
gc.collect()

cdwf_total_start = time.time()


# ================================================================
# STEP 1: BRIEF WARM-START
# ================================================================
print(f"[STEP 1] Brief Warm-Start ({EPOCHS_WARMSTART_BRIEF} epochs)")
print("-" * 70)

warmstart_start = time.time()
power_log = {"energy_J": 0.0, "last_t": warmstart_start}

BASE_CHANNELS = 64
USE_MAXPOOL = False
model = resnet50_1d(
    base=BASE_CHANNELS,
    use_maxpool=USE_MAXPOOL,
    num_classes=NUM_CLASSES,
).to(device)

if os.path.exists(BIAS_CKPT):
    try:
        print(f"Loading pre-trained checkpoint: {BIAS_CKPT}")
        state = torch.load(BIAS_CKPT, map_location="cpu")
        model.load_state_dict(state, strict=False)
        print("Loaded checkpoint successfully\n")
    except Exception as exc:
        print(f"Could not load checkpoint: {exc}")
        print("Training from random initialization\n")
else:
    print(f"Checkpoint not found: {BIAS_CKPT}")
    print("Training from random initialization\n")

model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_WARMSTART_BRIEF,
    pct_start=0.3,
    div_factor=10,
    final_div_factor=100,
)

for epoch in range(1, EPOCHS_WARMSTART_BRIEF + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device, power_log=power_log
    )
    val_loss, val_acc, val_auc = evaluate(model, val_loader, device)
    print(
        f"  Epoch {epoch}/{EPOCHS_WARMSTART_BRIEF}: "
        f"train={train_acc:.2f}% val={val_acc:.2f}% auc={val_auc:.2f}%"
    )

warmstart_time = time.time() - warmstart_start
warmstart_energy = power_log["energy_J"]
warmstart_auc = val_auc

print(f"\nWarm-start: {warmstart_auc:.2f}% AUC")
print(f"  Time: {warmstart_time:.1f}s")
print(f"  Energy: {warmstart_energy / 1000:.2f}kJ\n")


# ================================================================
# STEP 2: FEATURE EXTRACTION
# ================================================================
print("[STEP 2] Feature Extraction and Predictive Modeling")
print("-" * 70)

feature_start = time.time()

block_features = compute_block_features(model, val_loader, device, num_batches=10)
block_importance, max_gain, _ = build_predictor(
    block_features,
    warmstart_auc,
    fullft_auc,
)

feature_time = time.time() - feature_start
print(f"  Feature extraction time: {feature_time:.2f}s\n")


# ================================================================
# STEP 3: OPTIMIZATION
# ================================================================
print("[STEP 3] Constrained Optimization")
print("-" * 70)

optimization_start = time.time()

optimal_config = solve_optimal_architecture(
    block_features,
    block_importance,
    max_gain,
    warmstart_auc,
    model,
    fullft_auc,
    fullft_time,
    CONSTRAINT_TYPE,
    TARGET_SPEEDUP if CONSTRAINT_TYPE == "speedup" else TARGET_PARAMS_PCT,
)

optimization_time = time.time() - optimization_start
print(f"  Optimization time: {optimization_time:.3f}s\n")


# ================================================================
# STEP 4: APPLY ARCHITECTURE
# ================================================================
print("[STEP 4] Applying Optimal Architecture")
print("-" * 70)

apply_start = time.time()
blocks = get_block_dict(model)

for block_name in optimal_config["lora_blocks"]:
    block = blocks[block_name]

    for param in block.parameters():
        param.requires_grad = False

    if hasattr(block, "conv2") and not isinstance(block.conv2, LoRAConv1d):
        block.conv2 = LoRAConv1d(
            block.conv2,
            rank=optimal_config["lora_rank"],
            alpha=LORA_ALPHA,
        )

    print(f"  {block_name}: frozen + LoRA rank-{optimal_config['lora_rank']}")

for block_name in optimal_config["kept_blocks"]:
    block = blocks[block_name]
    for param in block.parameters():
        param.requires_grad = True
    print(f"  {block_name}: kept trainable")

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

apply_time = time.time() - apply_start

total_params, trainable_params, _ = count_params(model)
print("\n[Architecture Summary]")
print(f"  Trainable: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  Reduction: {total_params / trainable_params:.1f}x")
print(f"  Application time: {apply_time:.2f}s\n")


# ================================================================
# STEP 5: FINE-TUNING
# ================================================================
print(f"[STEP 5] Fine-Tuning ({EPOCHS_FINETUNE} epochs)")
print("-" * 70)

finetune_start = time.time()
power_log = {"energy_J": 0.0, "last_t": finetune_start}

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable, lr=MAX_LR * 0.3, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR * 0.3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_FINETUNE,
    pct_start=0.3,
    div_factor=10,
    final_div_factor=100,
)

best_val_auc = warmstart_auc
best_state = None

for epoch in range(1, EPOCHS_FINETUNE + 1):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device, power_log=power_log
    )
    val_loss, val_acc, val_auc = evaluate(model, val_loader, device)
    print(
        f"  Epoch {epoch}/{EPOCHS_FINETUNE}: "
        f"train={train_acc:.2f}% val={val_acc:.2f}% auc={val_auc:.2f}%"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)

finetune_time = time.time() - finetune_start
finetune_energy = power_log["energy_J"]

print(f"\nFinal val AUC: {best_val_auc:.2f}%")
print(f"  Fine-tuning time: {finetune_time:.1f}s")
print(f"  Fine-tuning energy: {finetune_energy / 1000:.2f}kJ\n")


# ================================================================
# TOTAL TIME AND BREAKDOWN
# ================================================================
cdwf_total_time = time.time() - cdwf_total_start
cdwf_total_energy = warmstart_energy + finetune_energy

print("[TIMING AND ENERGY BREAKDOWN]")
print(
    f"  Warm-start:    {warmstart_time:6.1f}s "
    f"({100 * warmstart_time / cdwf_total_time:5.1f}%)  "
    f"Energy: {warmstart_energy / 1000:5.2f}kJ"
)
print(
    f"  Features:      {feature_time:6.1f}s "
    f"({100 * feature_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Optimization:  {optimization_time:6.1f}s "
    f"({100 * optimization_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Apply arch:    {apply_time:6.1f}s "
    f"({100 * apply_time / cdwf_total_time:5.1f}%)"
)
print(
    f"  Fine-tuning:   {finetune_time:6.1f}s "
    f"({100 * finetune_time / cdwf_total_time:5.1f}%)  "
    f"Energy: {finetune_energy / 1000:5.2f}kJ"
)
print("  ----------------------------------------")
print(
    f"  TOTAL CDWF:    {cdwf_total_time:6.1f}s (100.0%)  "
    f"Energy: {cdwf_total_energy / 1000:5.2f}kJ\n"
)


# ================================================================
# FINAL EVALUATION
# ================================================================
print("=" * 70)
print("  FINAL RESULTS")
print("=" * 70 + "\n")

_, test_acc, test_auc = evaluate(model, test_loader, device)

retention = test_auc / fullft_auc
param_reduction = total_params / trainable_params
time_speedup = fullft_time / cdwf_total_time
energy_reduction = fullft_energy / cdwf_total_energy

print(f"Predicted AUC: {optimal_config['predicted_acc']:.2f}%")
print(f"Actual Test AUC: {test_auc:.2f}%")
print(f"Prediction Error: {abs(test_auc - optimal_config['predicted_acc']):.2f}%\n")

print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test AUC: {test_auc:.2f}%")
print(f"Retention: {retention * 100:.1f}%")
print(f"Trainable: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Parameter Reduction: {param_reduction:.1f}x\n")

print(f"Training Time: {cdwf_total_time:.1f}s")
print(f"Time Speedup: {time_speedup:.2f}x")
print(f"Target Speedup: {TARGET_SPEEDUP:.2f}x")
print(f"Constraint Met: {'Yes' if time_speedup >= TARGET_SPEEDUP * 0.9 else 'No'}\n")

print(f"Energy Reduction (measured): {energy_reduction:.2f}x")
print(f"  Full-FT energy: {fullft_energy / 1000:.2f}kJ")
print(f"  CDWF energy: {cdwf_total_energy / 1000:.2f}kJ\n")

print("=" * 70)
print("  COMPARISON: FULL_FT VS CDWF")
print("=" * 70)
print("Performance:")
print(f"  Full_FT: {fullft_auc:.2f}% AUC")
print(f"  CDWF:    {test_auc:.2f}% AUC ({retention * 100:.1f}% retention)")
print("\nEfficiency:")
print(f"  Parameters: {param_reduction:.1f}x fewer")
print(f"  Time:       {time_speedup:.2f}x faster (target: {TARGET_SPEEDUP:.2f}x)")
print(f"  Energy:     {energy_reduction:.2f}x less")
print("\nPrediction:")
print(f"  Predicted:  {optimal_config['predicted_acc']:.2f}%")
print(f"  Actual:     {test_auc:.2f}%")
print(f"  Error:      {abs(test_auc - optimal_config['predicted_acc']):.2f}%")
print("=" * 70 + "\n")


# ================================================================
# SAVE RESULTS
# ================================================================
results = {
    "method": "CDWF",
    "dataset": "spike",
    "constraint_type": CONSTRAINT_TYPE,
    "target_value": TARGET_SPEEDUP if CONSTRAINT_TYPE == "speedup" else TARGET_PARAMS_PCT,
    "fullft_auc": fullft_auc,
    "test_acc": test_acc,
    "test_auc": test_auc,
    "retention": retention,
    "trainable_params": trainable_params,
    "trainable_percent": 100 * trainable_params / total_params,
    "param_reduction": param_reduction,
    "training_time_s": cdwf_total_time,
    "time_speedup": time_speedup,
    "energy_reduction": energy_reduction,
    "warmstart_energy_J": warmstart_energy,
    "finetune_energy_J": finetune_energy,
    "total_energy_J": cdwf_total_energy,
    "predicted_auc": optimal_config["predicted_acc"],
    "prediction_error": abs(test_auc - optimal_config["predicted_acc"]),
    "n_kept_blocks": len(optimal_config["kept_blocks"]),
    "n_lora_blocks": len(optimal_config["lora_blocks"]),
    "lora_rank": optimal_config["lora_rank"],
}

df = pd.DataFrame([results])
if os.path.exists(RESULT_CSV):
    df = pd.concat([pd.read_csv(RESULT_CSV), df], ignore_index=True)
df.to_csv(RESULT_CSV, index=False)

print(f"Results saved to {RESULT_CSV}\n")

print("=" * 70)
print("  CDWF - SPIKE DETECTION COMPLETE")
print("=" * 70)
print("Constraint-driven architecture for PV cybersecurity")
print("Automatic optimization with no manual tuning")
print("Generalization to time-series data completed")
print("=" * 70)